# 实验 25 — dbt_utils 实战

**目标**：把 `dbt_utils` 包里最常用的几个宏过一遍：

- `generate_surrogate_key` —— 给 fact 表造稳定的 PK
- `date_spine` —— 生成连续日期序列，补全缺失日
- `pivot` —— 长表转宽表
- `unique_combination_of_columns` —— 已经在 _marts.yml 用过
- `star` —— 简化 select * except

[packages.yml](../dbt_project/packages.yml) 里已有 dbt_utils，`make deps` 装好的话直接用。

In [ ]:
import subprocess, os, duckdb
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

# 1) generate_surrogate_key：建一个临时 model
from pathlib import Path
m = Path('../dbt_project/models/marts/fct_rates_with_sk.sql')
m.write_text("""select
    {{ dbt_utils.generate_surrogate_key(['rate_date','currency']) }} as fact_id,
    rate_date,
    currency,
    rate
from {{ ref('stg_ecb_rates') }}
""")
print(dbt('run','--select','fct_rates_with_sk')[-300:])

In [ ]:
con = duckdb.connect('../data/sandbox.duckdb')
con.execute('select * from main.fct_rates_with_sk limit 3').df()

In [ ]:
# 2) date_spine：生成 2024 全年日期
spine = Path('../dbt_project/models/marts/dim_dates.sql')
spine.write_text("""{{ config(materialized='table') }}
with spine as (
    {{ dbt_utils.date_spine(
        datepart='day',
        start_date="cast('2024-01-01' as date)",
        end_date="cast('2025-01-01' as date)"
    ) }}
)
select date_day as the_date from spine
""")
print(dbt('run','--select','dim_dates')[-300:])
print('rows:', con.execute('select count(*) from main.dim_dates').fetchone()[0])
print(con.execute('select * from main.dim_dates order by 1 limit 3').df())

In [ ]:
# 用 date_spine left join 出“没汇率的日期”
con.execute("""
    select d.the_date
    from main.dim_dates d
    left join (select distinct rate_date from main.fct_daily_rates) f on d.the_date = f.rate_date
    where f.rate_date is null and d.the_date < current_date
    order by 1
    limit 10
""").df()

In [ ]:
# 3) pivot：长表转宽表（每个货币一列）
pivot = Path('../dbt_project/models/marts/rates_pivot.sql')
pivot.write_text("""select
    rate_date,
    {{ dbt_utils.pivot(
        'currency',
        ['USD','GBP','CHF','JPY','CNY'],
        agg='max',
        then_value='rate',
        else_value='null'
    ) }}
from {{ ref('stg_ecb_rates') }}
group by rate_date
order by rate_date
""")
print(dbt('run','--select','rates_pivot')[-300:])
con.execute('select * from main.rates_pivot limit 5').df()

In [ ]:
# 4) star: 'select * except'。在 staging 透传时常用
star = Path('../dbt_project/models/staging/stg_rates_clean.sql')
star.write_text("""select
    {{ dbt_utils.star(from=ref('stg_ecb_rates'), except=['loaded_at_load_id']) }}
from {{ ref('stg_ecb_rates') }}
""")
print(dbt('compile','--select','stg_rates_clean')[-200:])
p = Path('../dbt_project/target/compiled/dlt_dbt_sandbox/models/staging/stg_rates_clean.sql')
print(p.read_text())

In [ ]:
# 清理
import os
for f in ['../dbt_project/models/marts/fct_rates_with_sk.sql',
          '../dbt_project/models/marts/dim_dates.sql',
          '../dbt_project/models/marts/rates_pivot.sql',
          '../dbt_project/models/staging/stg_rates_clean.sql']:
    if os.path.exists(f): os.remove(f)
for t in ['fct_rates_with_sk','dim_dates','rates_pivot','stg_rates_clean']:
    try: con.execute(f'drop table if exists main.{t}; drop view if exists main.{t}')
    except: pass

## 生产环境踩坑提示

- **`generate_surrogate_key` 用 MD5**：稳定但碰撞理论存在。Snowflake / BigQuery 现代版的 `generate_surrogate_key` 已可选 `sha256`，但要看你 dbt 版本。
- **null 处理**：默认会把 null 替换成空字符串再 hash —— 这样两个不同的“null + xxx” 会撞 hash。重要 SK 要先 coalesce 成有意义的 sentinel。
- **`date_spine` 在 DuckDB**：默认实现使用 cross join，超大区间（10 年 + 小时粒度）会爆。生产用 generate_series 之类的引擎原生 API。
- **`pivot` 列名硬编码**：要让列动态生成，需要配合实验 18 的 `run_query` macro 在 parse 时查数据。
- **`star`**：除了 except 还有 `prefix` / `suffix` / `quote_identifiers`，重命名也能搞定，省去一长串 select 列。
- **包升级要看 CHANGELOG**：dbt-labs/dbt_utils 主版本升级偶尔改 signature，CI 别没 pin 版本。

## 思考题

- `generate_surrogate_key` vs 用 row_number() 的代理键，怎么权衡？哪个适合需要 cross-system 一致的场景？
- 用 `date_spine` + left join 写一个 “每天每个货币 1 行（缺失填 null）” 的稠密事实表。
- 探索一下 `dbt_utils.deduplicate`、`dbt_utils.equality`（用来对比两个表内容）、`dbt_utils.get_column_values`（动态拿列值）。